In [25]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [55]:
from mrc.io import get_tomo, save_tomo
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.measure import label, regionprops
from scipy.ndimage import binary_erosion, binary_dilation, binary_closing
from skimage.morphology import ball
import json
from scipy.spatial.distance import cdist
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import networkx as nx

In [ ]:
# def find_peak(semantic_mask):
#     distance_map = ndimage.distance_transform_edt(semantic_mask)
#     local_maxi = peak_local_max(distance_map, min_distance=4, threshold_abs=0.2, exclude_border=False)
#     return distance_map, local_maxi

def generate_distance_map(semantic_mask):
    distance_map = ndimage.distance_transform_edt(semantic_mask)
    return distance_map

def generate_local_maxi(semantic_mask):
    local_maxi = peak_local_max(semantic_mask, min_distance=4, threshold_abs=0.2, exclude_border=False)
    return local_maxi

In [28]:
semantic_path = f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/actin.mrc'
semantic_mask = get_tomo(semantic_path)
# distance_map, local_maxi = find_peak(semantic_mask)

In [29]:
distance_map = generate_distance_map(semantic_mask)
save_tomo(distance_map, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/actin/distance_map.mrc')

In [30]:
# save_tomo(distance_map, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/distance_map.mrc')
# center_distance_map = np.zeros_like(semantic_mask, dtype=np.int8)
# center_distance_map[distance_map > 4] = 1
# instance_label = label(center_distance_map)
# save_tomo(instance_label, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT/instance_label.mrc')
# instance_label = label(center_distance_map)

In [31]:
# flat = instance_label.ravel()

# # 1) 统计每个标签的像素数（比 unique + counts 更快）
# counts = np.bincount(flat)
# # 2) 挑出“像素数 >= 300” 的大标签
# large_labels = np.nonzero(counts >= 300)[0]

# # 3) 一次性保留大标签，其它全部置 0
# #    用映射表：下标是原标签，值是要映射到的新标签（不是大标签就映射到 0）
# mapping = np.zeros_like(counts, dtype=instance_label.dtype)
# mapping[large_labels] = large_labels
# # 4) 重映射一把，底层 C 实现，速度极快
# flat[:] = mapping[flat]

# new_instance_label = flat.reshape(instance_label.shape)

# save_tomo(new_instance_label, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT/new_instance_label.mrc')


In [32]:
# new_distance_map = generate_distance_map(new_instance_label)
# save_tomo(new_distance_map, f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/MT/new_distance_map.mrc')
local_maxi = generate_local_maxi(distance_map)
print(local_maxi)
# print(distance_map.shape)

[[ 70 902 418]
 [ 71 881 421]
 [ 74 791 559]
 ...
 [208 510 752]
 [213 462 734]
 [214 465 733]]


In [33]:
markers = np.zeros_like(semantic_mask, dtype=int)
for i, peak in enumerate(local_maxi):
    markers[tuple(peak)] = i + 1  # 标记每个局部最大值为唯一的实例ID
save_tomo(markers, 
          '/media/liushuo/data1/data/synapse_seg/pp463/Prediction/actin/keypoints.mrc')

In [34]:
def euclidean_distance_matrix(points):
    """
    计算三维空间中所有点之间的欧几里得距离，并返回一个距离矩阵
    """
    diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
    dist_matrix = np.linalg.norm(diff, axis=2)
    return dist_matrix

def tsp_3d(points):
    """
    基于贪心算法计算在三维空间中按最短距离连接点的顺序，返回排序后的points列表
    尝试每个点作为起点，找到最短的路径。
    """
    n = len(points)
    dist_matrix = euclidean_distance_matrix(points)
    
    # 初始化最小路径和排序
    min_total_length = float('inf')
    best_order = []

    # 尝试每个点作为起点
    for start_idx in range(n):
        visited = [False] * n
        order = []
        current_idx = start_idx
        visited[current_idx] = True
        order.append(points[current_idx])
        total_length = 0

        for _ in range(n - 1):
            # 当前点到其他未访问点的距离
            dist_to_current = dist_matrix[current_idx]
            
            # 选择最近的未访问点
            nearest_idx = np.argmin([dist_to_current[i] if not visited[i] else np.inf for i in range(n)])
            
            # 更新路径
            current_idx = nearest_idx
            visited[current_idx] = True
            order.append(points[current_idx])
            total_length += dist_to_current[current_idx]

        # 比较得到最小路径
        if total_length < min_total_length:
            min_total_length = total_length
            best_order = order

    return best_order

In [39]:
def split_keypoints_by_dbscan(keypoints, eps=5, min_samples=3):
    """
    用 DBSCAN 基于空间距离对点集做聚类，自动把粘连的多根细丝分开。

    Args:
        keypoints (List[np.ndarray] or np.ndarray of shape (N,3)): 原始三维点集合。
        eps (float): DBSCAN 邻域半径，视数据间距调整。
        min_samples (int): 最少点数阈值，低于该阈值的簇会被标记为噪声。

    Returns:
        Dict[int, np.ndarray]: key 为聚类标签 (从 1 开始)，value 为该簇的点集 (M,3)。
    """
    coords = np.vstack(keypoints)  # (N,3)
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(coords)
    labels = clustering.labels_
    clusters = {}
    for lbl in set(labels):
        if lbl == -1:
            continue  # 噪声
        pts = coords[labels == lbl]
        clusters[lbl + 1] = pts.astype(int)
    return clusters

In [54]:
def split_filaments_by_graph(
    keypoints: np.ndarray,
    radius: float = None,
    k_for_radius: int = 6,
    min_size: int = 3
):
    """
    将粘连在一起的多条曲线（细丝）关键点沿分叉处拆分。

    Args:
        keypoints (np.ndarray[N,3]): 原始的三维点集合。
        radius (float, optional): 邻域半径，若 None 则自动估计为 k 最近邻距离中位数 * 1.5。
        k_for_radius (int): 用于估计半径的 k 值。
        min_size (int): 丢弃点数 < min_size 的小片段。

    Returns:
        List[np.ndarray[M_i,3]]: 拆分后的若干段，每段为一个 (M_i,3) 的点阵。
    """
    coords = np.asarray(keypoints, dtype=float)
    N = len(coords)
    if N == 0:
        return []

    # 1) 如果没给 radius，就估计一个「相邻距离」的中位数 * 1.5
    if radius is None:
        nbrs = NearestNeighbors(n_neighbors=min(k_for_radius+1, N)).fit(coords)
        dists, _ = nbrs.kneighbors(coords)
        # dists[:,0] 是自己到自己的零距离，所以取第1列到第k列
        median_nn = np.median(dists[:, 1:], axis=1).mean()
        radius = median_nn * 1.5

    # 2) 构造半径邻域图
    nbrs = NearestNeighbors(radius=radius).fit(coords)
    adj = nbrs.radius_neighbors_graph(coords, mode='connectivity').tolil()

    G = nx.Graph()
    G.add_nodes_from(range(N))
    rows, cols = adj.nonzero()
    for i, j in zip(rows, cols):
        if i < j:
            G.add_edge(i, j)

    # 3) 找 articulation points（切割点）
    cut_nodes = list(nx.articulation_points(G))
    # 也可以只用度 > 2：  cut_nodes = [n for n, d in G.degree() if d > 2]

    # 4) 移除这些节点
    G_removed = G.copy()
    G_removed.remove_nodes_from(cut_nodes)

    # 5) 取每个连通分量，收集原始坐标
    segments = []
    for comp in nx.connected_components(G_removed):
        if len(comp) >= min_size:
            pts = coords[list(comp)]
            segments.append(pts.astype(int))  # 转回整数坐标
    return segments

In [60]:
def find_instance_for_keypoints(instance_mask, keypoints_list):
    """
    根据instance_mask为每个关键点分配对应的实例ID
    """
    instance_ids = []
    for kp in keypoints_list:
        z, y, x = kp
        instance_id = instance_mask[int(z), int(y), int(x)]
        instance_ids.append(instance_id)
    return instance_ids

def sort_keypoints_by_distance(keypoints):
    """
    根据最短距离排序关键点，使点从头到尾连成的距离最短
    使用贪心算法，选择最近的点连接。
    """
    # 初始化第一个点
    sorted_keypoints = [keypoints[0]]
    keypoints_remaining = set(map(tuple, keypoints[1:]))  # 剩余的点
    current_point = keypoints[0]

    while keypoints_remaining:
        # 计算当前点到剩余点的距离
        distances = cdist([current_point], list(keypoints_remaining))
        # 找到最近的点
        next_point = list(keypoints_remaining)[np.argmin(distances)]
        sorted_keypoints.append(next_point)
        keypoints_remaining.remove(tuple(next_point))
        current_point = next_point
    
    return np.array(sorted_keypoints)

def calculate_total_length(sorted_keypoints):
    """
    计算按最短路径排序后的关键点总长度
    """
    length = 0
    for i in range(1, len(sorted_keypoints)):
        length += np.linalg.norm(sorted_keypoints[i] - sorted_keypoints[i - 1])
    return length

def save_results_to_json(instances_info, output_json_path):
    """
    将实例信息保存到JSON文件
    """
    with open(output_json_path, 'w') as f:
        json.dump(instances_info, f, indent=4)

def process_instance_masks(instance_mask, keypoints_list, output_json_path):
    """
    处理实例mask和关键点，生成最终的JSON结果
    """
    # 1. 为每个关键点分配实例ID
    instance_ids = find_instance_for_keypoints(instance_mask, keypoints_list)

    # 2. 按实例ID分组关键点
    instance_keypoints = {}
    for idx, instance_id in enumerate(instance_ids):
        instance_keypoints.setdefault(instance_id, []).append(keypoints_list[idx])

    # 2.1 删除 ID 为 0 的组（通常代表背景）
    instance_keypoints.pop(0, None)

    # 2.2 丢弃点数少于 3 的组
    instance_keypoints = {
        inst_id: pts
        for inst_id, pts in instance_keypoints.items()
        if len(pts) >= 3
    }

    # 3. 对每个实例的关键点按最短路径排序，并收集信息
    instances_info = []
    # 按原始 ID 排序（可选），然后重新编号
    for new_id, (orig_id, keypoints) in enumerate(sorted(instance_keypoints.items()), start=1):
        # 假设 tsp_3d 就是 sort_keypoints_by_distance
        sorted_kps = tsp_3d(np.array(keypoints))  
        total_length = calculate_total_length(sorted_kps)

        instances_info.append({
            "id": new_id,  # 重新从 1 开始编号
            # "orig_id": int(orig_id),  # 如果需要保留原始 ID，可以加上这行
            "points": [[float(z), float(y), float(x)] for z, y, x in sorted_kps],
            "length": float(total_length)
        })
        
        if new_id==14:
            all_pts = np.vstack(sorted_kps)
            segments = split_filaments_by_graph(all_pts, radius=3, min_size=3)
            for idx, seg in enumerate(segments, 1):
                print(f"Segment {idx}: {seg.shape[0]} points")
            # for cid, pts in clusters.items():
            #     markers = np.zeros_like(semantic_mask, dtype=int)
            #     print(f"Cluster {cid}: {pts.shape[0]} pts")
            #     for i, peak in enumerate(pts):
            #         markers[tuple(peak)] = i + 1  # 标记每个局部最大值为唯一的实例ID
            #     save_tomo(markers, 
            #         f'/media/liushuo/data1/data/synapse_seg/pp463/Prediction/actin/{cid}.mrc')
                    
                

    # 4. 保存结果到 JSON 文件
    save_results_to_json(instances_info, output_json_path)

In [63]:
output_json_path = '/media/liushuo/data1/data/synapse_seg/pp463/Prediction/actin/actin_point.json'
instance_label = label(semantic_mask)
process_instance_masks(instance_label, local_maxi, output_json_path)


[]
[]
